- 稳态分布 $\pi$ 需满足方程 $\pi P = \pi$ 且 $\sum \pi_i = 1$。设 $\pi = [\pi_A, \pi_B, \pi_C, \pi_D, \pi_E]$。
- 存在性，
    - 有限状态马尔可夫链，总是至少存在一个稳态分布。
    - 转移矩阵 $P$ 是随机矩阵（行和为 1）。根据 Perron-Frobenius 定理，随机矩阵一定有一个等于 1 的特征值（$\lambda=1$）。这个特征值对应的特征向量就是稳态分布。
- 唯一性
    - 如果有多个特征值等于 1，说明稳态分布不唯一
    - 在马尔可夫链理论中，特征值 $\lambda=1$ 的个数对应于“互不相通的闭合集”的数量。
- “存在稳态分布”不等于“系统会收敛”。

### case 1：存在且唯一

$$
P = \begin{bmatrix}
0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 0.5 & 0 & 0.5 & 0 \\
0 & 0 & 1 & 0 & 0 \\
0 & 0.1 & 0 & 0 & 0.9
\end{bmatrix}
$$

- 列的方向
    - $\pi_A = 0$ (A列全为0，且无其他状态流向A)
    - $\pi_B = 0.5\pi_C + 0.1\pi_E$
    - $\pi_C = \pi_A + \pi_D$
        - $\pi_C=\pi_D$
    - $\pi_D = \pi_B + 0.5\pi_C$
    - $\pi_E = 0.9\pi_E$
        - $\pi_E=0$
- $\pi = [0, 0.2, 0.4, 0.4, 0]$

In [13]:
import numpy as np

P = np.array([
    [0,   0,   1,   0,   0],
    [0,   0,   0,   1,   0],
    [0, 0.5,   0, 0.5,   0],
    [0,   0,   1,   0,   0],
    [0, 0.1,   0,   0, 0.9]
])

In [14]:
# 方法一：特征值分解法 (最通用标准的方法)
# 求 P.T 的特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(P.T)

# 找到接近于 1 的特征值的索引
# np.isclose 用于处理浮点数精度问题
idx = np.argmin(np.abs(eigenvalues - 1))

# 提取对应的特征向量 (此时是列向量)
pi = eigenvectors[:, idx]

# 归一化 (让概率之和为 1)
pi = pi / pi.sum()

# 取实部 (去除计算过程中可能产生的极小虚部数值误差)
pi = pi.real

print(eigenvalues)
print("稳态分布 (特征值法):", pi)

[ 1. +0.j  -0.5+0.5j -0.5-0.5j  0. +0.j   0.9+0.j ]
稳态分布 (特征值法): [-0.   0.2  0.4  0.4 -0. ]


In [7]:
# 方法二：矩阵幂迭代法 (直观，模拟长期运行)
# 也就是计算 P 的 n 次方，当 n 很大时，每一行都趋近于稳态分布
n = 1000
P_n = np.linalg.matrix_power(P, n)
pi_power = P_n[0]

### 存在不唯一

$$
P = \begin{bmatrix}
0 & 0.5 & 0 & 0.5 \\
0 & 0 & 1 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

- 稳态分布存在，但 不唯一 (Not Unique)。

In [10]:
# 定义转移矩阵
P = np.array([
    [0, 0.5, 0, 0.5],
    [0, 0,   1, 0  ],
    [0, 1,   0, 0  ],
    [0, 0,   0, 1  ]
])
# 验证方法 1: 查看特征值
# 如果有多个特征值等于 1，说明稳态分布不唯一
eigenvalues, eigenvectors = np.linalg.eig(P.T)
print("特征值:", np.round(eigenvalues, 2))

特征值: [ 1. -1.  1.  0.]


- 这里有两个 1（`[ 1. -1.  1.  0.]`），说明系统有两个独立的“归宿”（一个是 {B,C} 循环圈，一个是 {D} 自环）。

In [12]:
# 验证方法 2: 查看矩阵的大数值次幂 (模拟长期转移)
# 注意：由于 B-C 是周期性的（你给我，我给你），奇数次幂和偶数次幂会不同
# 我们看一个大的偶数次幂，观察"行"是否相同
P_100 = np.linalg.matrix_power(P, 100)
print("\nP^100 (长期转移概率矩阵):\n", P_100)

P_101 = np.linalg.matrix_power(P, 101)
print("\nP^100 (长期转移概率矩阵):\n", P_101)


P^100 (长期转移概率矩阵):
 [[0.  0.  0.5 0.5]
 [0.  1.  0.  0. ]
 [0.  0.  1.  0. ]
 [0.  0.  0.  1. ]]

P^100 (长期转移概率矩阵):
 [[0.  0.5 0.  0.5]
 [0.  0.  1.  0. ]
 [0.  1.  0.  0. ]
 [0.  0.  0.  1. ]]


```
[[0.  0.  0.5 0.5]   <-- 从 A 出发：50% 陷入 {B,C}，50% 陷入 D
 [0.  1.  0.  0. ]   <-- 从 B 出发：100% 在 {B,C} 里 (偶数步在B)
 [0.  0.  1.  0. ]   <-- 从 C 出发：100% 在 {B,C} 里 (偶数步在C)
 [0.  0.  0.  1. ]]  <-- 从 D 出发：100% 在 D 里
```

### 不收敛

- 如果一个链是 周期的 (Periodic)（例如 A -> B -> A -> B...）

$$
[\pi_A, \pi_B] \times \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} = [\pi_A, \pi_B]
$$

In [18]:
# 定义转移矩阵
P = np.array([
    [0, 1],
    [1, 0],
])
eigenvalues, eigenvectors = np.linalg.eig(P.T)
print("特征值:", np.round(eigenvalues, 2))

P_100 = np.linalg.matrix_power(P, 100)
print("\nP^100 (长期转移概率矩阵):\n", P_100)
P_101 = np.linalg.matrix_power(P, 101)
print("\nP^101 (长期转移概率矩阵):\n", P_101)

特征值: [ 1. -1.]

P^100 (长期转移概率矩阵):
 [[1 0]
 [0 1]]

P^101 (长期转移概率矩阵):
 [[0 1]
 [1 0]]
